#### importación de librerias


In [ ]:
import pandas as pd
import os
from datetime import datetime

#### Exploración e ingenieria de datos

In [ ]:


dataset = pd.read_csv(
    "~/Documentos/proyectos/microred/data/raw/household_power_consumption.txt", 
    sep=";",
    na_values=["?"]
    )

#conbinar celdas para formar un datetime
dataset["DateTime"] = dataset["Date"] + " " + dataset["Time"]

# transformar el valor a datetime
dataset["DateTime"] = pd.to_datetime(dataset["DateTime"])


# eliminar columnas no necearias

dataset = dataset.drop(columns=["Date", "Time"])


# creacion de columnas como caracteristicas nuevas
dataset["Hour"]  = dataset["DateTime"].dt.hour
dataset["Date_Of_Week"] = dataset["DateTime"].dt.dayofweek
dataset["Month"] = dataset["DateTime"].dt.month


print(dataset.columns)



Index(['Global_active_power', 'Global_reactive_power', 'Voltage',
       'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3', 'DateTime', 'Hour', 'Date_Of_Week', 'Month'],
      dtype='str')


/tmp/ipykernel_645729/1629509364.py:15: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dataset["DateTime"] = pd.to_datetime(dataset["DateTime"])


Tipos de datos de el dataset

In [152]:
print(dataset.dtypes)

Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
DateTime                 datetime64[us]
Hour                              int32
Date_Of_Week                      int32
Month                             int32
dtype: object


Cambiar el indice del dataset

In [153]:

# ejecutar solo una vez
dataset = dataset.set_index("DateTime")



In [154]:
print(dataset.index)

DatetimeIndex(['2006-12-16 17:24:00', '2006-12-16 17:25:00',
               '2006-12-16 17:26:00', '2006-12-16 17:27:00',
               '2006-12-16 17:28:00', '2006-12-16 17:29:00',
               '2006-12-16 17:30:00', '2006-12-16 17:31:00',
               '2006-12-16 17:32:00', '2006-12-16 17:33:00',
               ...
               '2010-11-26 20:53:00', '2010-11-26 20:54:00',
               '2010-11-26 20:55:00', '2010-11-26 20:56:00',
               '2010-11-26 20:57:00', '2010-11-26 20:58:00',
               '2010-11-26 20:59:00', '2010-11-26 21:00:00',
               '2010-11-26 21:01:00', '2010-11-26 21:02:00'],
              dtype='datetime64[us]', name='DateTime', length=2075259, freq=None)


Interpolar valores nulos

In [155]:
# interpolación de valores nulos

def interpolar(x, x0, y0, x1, y1):
    return y0 + ((y1-y0)/(x1-x0))*(x-x0)


dataset.interpolate(method='time', inplace=True)



,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Hour,Date_Of_Week,Month
DateTime,,,,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,17,5,12
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,17,5,12
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,17,5,12
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,17,5,12
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,17,5,12
...,...,...,...,...,...,...,...,...,...,...
2010-11-26 20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0,20,4,11
2010-11-26 20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0,20,4,11
2010-11-26 21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0,21,4,11


#### Eliminación de datos nulos residuales y downsampling de datos

Al aplicar la interpolación de datos podemos quedar on valores nulos al inicio o final lo cual genera errores puesto que se necesitan valores anteriores y finales, los cuales no estan

El dowsampling lo hacemos para reducir los pases en horas y no minutos





In [156]:

#eliminado de nulos en limites, donde no se pudo aplicar la interpolación
dataset.dropna(inplace=True)


# reducción de paso de minutos en horas
dataset_sampling = dataset.resample('h').mean()

#### Generación de columnas como hora, dia de la semana y mes






In [ ]:

dataset_sampling["Month"] = dataset_sampling.index.month
dataset_sampling["Hour"]  = dataset_sampling.index.hour
dataset_sampling["Date_Of_Week"] = dataset_sampling.index.dayofweek

#### Guardar a csv

In [ ]:
dataset_sampling.to_csv("~/Documentos/proyectos/microred/data/processed")